# CIFAR-10 Classical ML Benchmark


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ---------- Data ----------
from tensorflow.keras.datasets import cifar10

# ---------- Preprocessing ----------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ---------- Models ----------
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# ---------- Boosting ----------
from xgboost import XGBClassifier

# ---------- Evaluation ----------
from sklearn.metrics import accuracy_score

### 1. Load CIFAR-10


In [ ]:
print("Loading CIFAR-10 ...")
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

y_train = y_train.ravel()
y_test = y_test.ravel()

# Flatten images: 32x32x3 -> 3072
X_train = X_train.reshape(len(X_train), -1)
X_test = X_test.reshape(len(X_test), -1)

print(f"Training samples: {X_train.shape[0]}, \nTest samples: {X_test.shape[0]}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
print(f"Feature dimension: {X_train.shape[1]}")
print("Classes: 10 (CIFAR-10)")
print("Data loaded.\n")

Loading CIFAR-10 ...
Training samples: 50000, 
Test samples: 10000
y_train shape: (50000,), y_test shape: (10000,)
Feature dimension: 3072
Classes: 10 (CIFAR-10)
Data loaded.



### 2. Normalization


In [ ]:
print("Scaling features ...")
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Scaling features ...


### 3. Dimensionality Reduction (Critical)


In [ ]:
print("Applying PCA ...")
pca = PCA(n_components=300, random_state=42)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")
print(f"Training samples: {X_train.shape[0]}, \nTest samples: {X_test.shape[0]}")
print(f"Feature dimension: {X_train.shape[1]}")

Applying PCA ...
Explained variance: 0.965
Training samples: 50000, 
Test samples: 10000
Feature dimension: 300


### 4. Models


In [ ]:
models = {}

# Logistic Regression
models["Logistic"] = LogisticRegression(
    max_iter=1000,
    solver="saga",
    C=1.0,
    n_jobs=-1
)

# KNN
models["KNN"] = KNeighborsClassifier(
    n_neighbors=7,
    weights="distance",
    metric="euclidean",
    n_jobs=1
)

# Naive Bayes
models["NaiveBayes"] = GaussianNB()

# Decision Tree
models["DecisionTree"] = DecisionTreeClassifier(
    max_depth=20,
    min_samples_leaf=10,
    random_state=42
)

# Random Forest
models["RandomForest"] = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

# SVM (subset for sanity)
models["SVM"] = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

# MLP
models["MLP"] = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation="relu",
    solver="adam",
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    random_state=42
)

# XGBoost
models["XGBoost"] = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=10,
    n_jobs=-1,
    eval_metric="mlogloss"
)

### 5. Training & Evaluation


In [ ]:
print("\nTraining models...\n")

results = {}

for name, model in models.items():
    print(f"Training {name} ...")

    # SVM is too slow on full data
    if name == "SVM":
        model.fit(X_train[:15000], y_train[:15000])
        preds = model.predict(X_test)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(f"{name:12s} Accuracy: {acc:.4f}\n")


Training models...

Training Logistic ...
Logistic     Accuracy: 0.4089

Training KNN ...


  File "c:\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


KNN          Accuracy: 0.3703

Training NaiveBayes ...
NaiveBayes   Accuracy: 0.2951

Training DecisionTree ...
DecisionTree Accuracy: 0.2914

Training RandomForest ...
RandomForest Accuracy: 0.4670

Training SVM ...
SVM          Accuracy: 0.5000

Training MLP ...
MLP          Accuracy: 0.5313

Training XGBoost ...
XGBoost      Accuracy: 0.5287



### 6. Summary


In [ ]:
print("=" * 40)
print("Final Accuracy Summary")
print("=" * 40)

for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:12s}: {acc:.4f}")

Final Accuracy Summary
MLP         : 0.5313
XGBoost     : 0.5287
SVM         : 0.5000
RandomForest: 0.4670
Logistic    : 0.4089
KNN         : 0.3703
NaiveBayes  : 0.2951
DecisionTree: 0.2914


In [13]:
from model_check import evaluate_sklearn_models

models_list = []

for name, model in models.items():
    models_list.append((name, model))


evaluate_sklearn_models(models_list, X_test, y_test)


Evaluating model: Logistic
Test Accuracy: 0.4089

Confusion Matrix:



,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
airplane,491,41,56,36,21,27,27,55,181,65
automobile,66,478,25,36,25,39,34,55,74,168
bird,102,47,262,96,122,78,145,78,49,21
cat,39,55,101,267,61,187,129,49,42,70
deer,60,23,146,64,290,90,165,106,26,30
dog,42,41,102,163,76,349,84,69,44,30
frog,14,31,72,125,94,84,491,43,17,29
horse,51,45,67,63,82,84,47,447,36,78
ship,163,71,23,28,9,43,11,19,537,96
truck,69,176,24,23,19,25,47,47,93,477



Per-Class Performance:



,Class,Total,Correct,Incorrect,Accuracy (%)
0,airplane,1000,491,509,49.1
1,automobile,1000,478,522,47.8
2,bird,1000,262,738,26.2
3,cat,1000,267,733,26.7
4,deer,1000,290,710,29.0
5,dog,1000,349,651,34.9
6,frog,1000,491,509,49.1
7,horse,1000,447,553,44.7
8,ship,1000,537,463,53.7
9,truck,1000,477,523,47.7



Classification Report:



...

,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
airplane,600,38,53,32,27,15,26,23,123,63
automobile,35,626,17,24,10,15,21,30,56,166
bird,85,35,399,79,113,78,102,50,32,27
cat,33,33,88,341,59,189,120,55,30,52
deer,46,9,113,68,455,56,116,88,29,20
dog,21,18,70,218,71,404,59,84,32,23
frog,8,22,63,69,76,37,665,24,13,23
horse,32,24,40,65,71,84,35,548,20,81
ship,95,62,14,25,23,24,11,17,666,63
truck,44,166,13,34,13,17,27,38,65,583



Per-Class Performance:



,Class,Total,Correct,Incorrect,Accuracy (%)
0,airplane,1000,600,400,60.0
1,automobile,1000,626,374,62.6
2,bird,1000,399,601,39.9
3,cat,1000,341,659,34.1
4,deer,1000,455,545,45.5
5,dog,1000,404,596,40.4
6,frog,1000,665,335,66.5
7,horse,1000,548,452,54.8
8,ship,1000,666,334,66.6
9,truck,1000,583,417,58.3



Classification Report:



,precision,recall,f1-score,support
airplane,0.600601,0.6000,0.600300,1000.0000
automobile,0.606002,0.6260,0.615839,1000.0000
bird,0.458621,0.3990,0.426738,1000.0000
cat,0.357068,0.3410,0.348849,1000.0000
deer,0.495643,0.4550,0.474453,1000.0000
dog,0.439608,0.4040,0.421053,1000.0000
frog,0.562606,0.6650,0.609533,1000.0000
horse,0.572623,0.5480,0.560041,1000.0000
ship,0.624765,0.6660,0.644724,1000.0000
truck,0.529519,0.5830,0.554974,1000.0000
